# Task Router: baseline, fine-tune and evaluate

This notebook reproduces one training round of Task Router (v1, v2, v3 or v4) on a GPU runtime such as Google Colab. It:

1. gets this repository (clone, or a path you set) and checks every data file against the SHA-256 values in `data/manifest.json` and `data/rounds.json`;
2. installs the Laya framework at the pinned commit and downloads the pinned base model `convaiinnovations/laya-multilingual`;
3. scores the untuned base model on the validation split;
4. fine-tunes on the chosen round's training file with the recorded hyperparameters (switch, off by default);
5. scores the fine-tuned model on the same validation split, then on the test split (second switch, off by default).

All rows are synthetic. Nothing here calls an external AI service; executor names in the data (Muse, Alex, Orion, Nova, ...) are fictional inputs to classify.

Paths come from environment variables with Colab defaults: `TASK_ROUTER_REPO`, `TASK_ROUTER_REPO_URL`, `TASK_ROUTER_WORKDIR`, `TASK_ROUTER_MODEL_CACHE`, `TASK_ROUTER_ROUND`. No account, API key or Hugging Face token is needed.

## 1. Switches

Leave both off for the first pass. After the baseline has run and you have read its report, set `RUN_TRAINING = True` and re-run only this cell. Turn on `RUN_FINAL_TEST` only once the configuration is fixed: a test split that has been used to make decisions is no longer an unseen test.

In [ ]:
RUN_TRAINING = False    # set True after the baseline validation run
RUN_FINAL_TEST = False  # set True only once the configuration is fixed
print("training:", RUN_TRAINING, "| final test:", RUN_FINAL_TEST)

## 2. Configuration and a fresh run directory

Run this cell once per experiment. Every run writes to a new directory; old results are never overwritten.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys, uuid

ROUND = os.environ.get("TASK_ROUTER_ROUND", "v4")  # v1, v2, v3 or v4
REPO_URL = os.environ.get("TASK_ROUTER_REPO_URL", "https://github.com/Waytrine/task-router.git")
REPO_DIR = Path(os.environ.get("TASK_ROUTER_REPO", "/content/task-router"))
WORK_DIR = Path(os.environ.get("TASK_ROUTER_WORKDIR", "/content/task-router-runs"))
MODEL_CACHE = Path(os.environ.get("TASK_ROUTER_MODEL_CACHE", "/content/task-router-models"))

SEED = 20261026
LAYA_COMMIT = "b9fe351ce81e2d974bca2bf219c1e4030c0a05ea"
MODEL_ID = "convaiinnovations/laya-multilingual"
MODEL_REVISION = "1720e3e3357cfe1e281542e223f8273b0890ca34"
TRAIN_FILES = {"v1": "train.jsonl", "v2": "train_v2.jsonl", "v3": "train_v3.jsonl", "v4": "train_v4.jsonl"}
if ROUND not in TRAIN_FILES:
    raise ValueError(f"ROUND must be one of {sorted(TRAIN_FILES)}")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8] + "-" + ROUND
OUT = WORK_DIR / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
BASE_DIR = MODEL_CACHE / MODEL_REVISION
FINETUNED_DIR = OUT / f"task-router-{ROUND}"
print("run directory:", OUT)

## 3. Get the repository and verify the data

If `TASK_ROUTER_REPO` already holds a checkout it is used as is; otherwise the repository is cloned. Every split is checked against its published SHA-256, and the chosen training file must share no family, scenario group or message with validation, calibration or test.

In [ ]:
if not (REPO_DIR / "data" / "manifest.json").is_file():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
DATA = REPO_DIR / "data"
EVALUATOR = REPO_DIR / "training" / "evaluate.py"
TRAIN = DATA / TRAIN_FILES[ROUND]
VALIDATION = DATA / "validation.jsonl"
TEST = DATA / "test.jsonl"

manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
rounds = json.loads((DATA / "rounds.json").read_text(encoding="utf-8"))
provenance = json.loads((DATA / "provenance.json").read_text(encoding="utf-8"))
assert provenance["laya_source_commit"] == LAYA_COMMIT
assert provenance["base_model"] == MODEL_ID
assert provenance["base_model_revision"] == MODEL_REVISION
expected_hashes = dict(manifest["sha256"]) | {path: info["sha256"] for path, info in rounds["files"].items()}
for relative, digest in expected_hashes.items():
    actual = hashlib.sha256((REPO_DIR / relative).read_bytes()).hexdigest()
    if actual != digest:
        raise ValueError(f"SHA-256 mismatch for {relative}; re-download the repository without line-ending conversion")

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8-sig").splitlines() if line.strip()]

train_rows = read_jsonl(TRAIN)
for name in ("validation", "calibration", "test"):
    held_out = read_jsonl(DATA / f"{name}.jsonl")
    for key in ("family_id", "group_id"):
        shared = {row[key] for row in train_rows} & {row[key] for row in held_out}
        assert not shared, f"{key} shared between {TRAIN.name} and {name}: {sorted(shared)[:5]}"
    assert not {r["state"]["message"] for r in train_rows} & {r["state"]["message"] for r in held_out}
train_decisions = sum(len(row["expected"]) for row in train_rows)
print(f"{TRAIN.name}: {len(train_rows)} rows, {train_decisions} head-level training items")
(OUT / "data-hashes.json").write_text(json.dumps(expected_hashes, indent=2), encoding="utf-8")

## 4. Install the Laya framework at the pinned commit

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install",
                f"git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}"], check=True)

## 5. Check the GPU and download the pinned base model

The base model is public; no token is used.

In [ ]:
import torch
from huggingface_hub import snapshot_download

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. In Colab: Runtime > Change runtime type > GPU.")
print("GPU:", torch.cuda.get_device_name(0))
BASE_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, local_dir=str(BASE_DIR), token=False)
if not (BASE_DIR / "rl_agent_config.json").is_file():
    raise RuntimeError("The download does not look like a Laya checkpoint (no rl_agent_config.json).")

## 6. Record the environment

In [ ]:
environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "laya_commit": LAYA_COMMIT, "base_model": MODEL_ID, "base_revision": MODEL_REVISION,
    "round": ROUND, "train_file": TRAIN.name, "seed": SEED,
}
(OUT / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
(OUT / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
print(json.dumps(environment, indent=2))

## 7. Format check and training dry run

`laya-evals validate` checks each file's format; `laya-train --dry-run` shows how the rows expand into training items without updating any weights. Stop and fix any warning before training.

In [ ]:
def run_logged(args, logfile):
    logfile = Path(logfile)
    logfile.parent.mkdir(parents=True, exist_ok=True)
    with logfile.open("x", encoding="utf-8") as stream:
        process = subprocess.Popen([str(a) for a in args], cwd=str(REPO_DIR),
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
        result = process.wait()
    if result != 0:
        raise subprocess.CalledProcessError(result, args)

for path in (TRAIN, VALIDATION, TEST):
    run_logged(["laya-evals", "validate", path], OUT / f"validate-{path.stem}.log")
run_logged(["laya-train", "--data", TRAIN, "--base", BASE_DIR, "--dry-run"], OUT / "train-dry-run.log")

## 8. Baseline: the untuned base model on validation

Real GPU inference, no training. Every row's raw output, failures and a mistakes CSV are saved. The majority-class baseline is computed from the training file only.

In [ ]:
BASELINE_VALIDATION = OUT / "baseline-validation"
run_logged([sys.executable, EVALUATOR, "run", VALIDATION, "--model-dir", BASE_DIR,
            "--out", BASELINE_VALIDATION, "--train-data", TRAIN, "--device", "cuda"],
           OUT / "baseline-validation.log")
baseline = json.loads((BASELINE_VALIDATION / "report.json").read_text(encoding="utf-8"))
print("mean head accuracy:", baseline["mean_head_accuracy"], "| all-10 exact:", baseline["all_heads_exact_match"],
      "| failures:", baseline["failures"])

## 9. Fine-tune (switch)

The hyperparameters are the ones recorded for Task Router v1-v4: soft cross-entropy, 4 epochs, micro-batch 8 x gradient accumulation 2, encoder LR 2.5e-5, head LR 1e-4, weight decay 0.01, gradient clip 1.0, encoder not frozen. Mixed precision follows the checkpoint (bf16). Laya's internal calibration split is turned off (`calib_max=0`), because it samples individual head-level items and would mix questions from the same scenario across training and calibration.

In [ ]:
if not RUN_TRAINING:
    print("Training is off. Set RUN_TRAINING = True in section 1, re-run that cell, then this one.")
else:
    from laya.train import finetune, TrainConfig
    import gc
    if baseline["failures"]:
        raise RuntimeError("The baseline had inference failures; fix them before training.")
    if FINETUNED_DIR.exists():
        raise FileExistsError("Output directory exists; start a new run instead.")
    config_values = dict(loss="soft-ce", epochs=4, micro_batch=8, grad_accum=2,
                         encoder_lr=2.5e-5, head_lr=1e-4, weight_decay=0.01, grad_clip=1.0,
                         freeze_encoder=False, calib_max=0, calib_frac=0.0, seed=SEED)
    (OUT / "training-config.json").write_text(json.dumps(config_values, indent=2), encoding="utf-8")
    result = finetune(data=str(TRAIN), model_dir=str(BASE_DIR), output_dir=str(FINETUNED_DIR),
                      config=TrainConfig(**config_values), device="cuda")
    del result
    gc.collect()
    torch.cuda.empty_cache()
    if not (FINETUNED_DIR / "rl_agent_config.json").is_file():
        raise RuntimeError("Training did not leave a loadable checkpoint; check the log before evaluating.")
    (OUT / "training-completed.json").write_text(json.dumps({
        "finished_at_utc": datetime.now(timezone.utc).isoformat(), "round": ROUND,
        "train_sha256": expected_hashes[f"data/{TRAIN.name}"]}, indent=2), encoding="utf-8")
    print("checkpoint:", FINETUNED_DIR)

## 10. Same validation split, before and after

Besides the mean, read macro F1, per-label recall, the Chinese/English split and `mistakes.csv`.

In [ ]:
FINETUNED_VALIDATION = OUT / "finetuned-validation"
if not (OUT / "training-completed.json").is_file():
    print("No completed training in this run; skipping.")
else:
    run_logged([sys.executable, EVALUATOR, "run", VALIDATION, "--model-dir", FINETUNED_DIR,
                "--out", FINETUNED_VALIDATION, "--train-data", TRAIN, "--device", "cuda"],
               OUT / "finetuned-validation.log")
    run_logged([sys.executable, EVALUATOR, "compare", BASELINE_VALIDATION / "report.json",
                FINETUNED_VALIDATION / "report.json", "--out", OUT / "validation-comparison.md"],
               OUT / "validation-comparison.log")

## 11. Final test (switch)

Both checkpoints are scored on the same test file. Once you have looked at test errors and changed anything because of them, report later test numbers as regression numbers, not as an unseen test.

In [ ]:
if not RUN_FINAL_TEST:
    print("Final test is off. Set RUN_FINAL_TEST = True in section 1 once the configuration is fixed.")
else:
    if not (FINETUNED_VALIDATION / "report.json").is_file():
        raise RuntimeError("Finish training and the validation comparison first.")
    for label, checkpoint in (("baseline-test", BASE_DIR), ("finetuned-test", FINETUNED_DIR)):
        run_logged([sys.executable, EVALUATOR, "run", TEST, "--model-dir", checkpoint, "--out", OUT / label,
                    "--train-data", TRAIN, "--device", "cuda"], OUT / f"{label}.log")
    run_logged([sys.executable, EVALUATOR, "compare", OUT / "baseline-test/report.json",
                OUT / "finetuned-test/report.json", "--out", OUT / "test-comparison.md"],
               OUT / "test-comparison.log")

## 12. Package the results

The zip contains the environment record, logs, per-row predictions, mistakes, comparisons and (if trained) the fine-tuned checkpoint. The public base model is not included.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK_DIR / f"task-router-results-{RUN_ID}"), "zip",
                              root_dir=str(OUT.parent), base_dir=OUT.name)
print("results:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## Not covered here

- Temperature scaling. For v4 a temperature was fitted after training on `data/calibration_v4.jsonl` (60 rows; T = 3.51, ECE 0.040 to 0.015 on that same split). That step is not part of this notebook.
- The v5 rounds, which also used reviewed real requests. Those rows are private and are not in this repository.